# Tutorial 05: Wikipedia API and data cleaning

Author: Maximilian Kreutner

In this notebook we will find all members of the current european parliament using wikidata.

Then, we will use the [python wrapper around the Wikipedia-API](https://github.com/martin-majlis/Wikipedia-API) to download their full biographies, clean the text, and extract their social media usernames.

In [ ]:
# Install dependencies
!pip install Wikipedia-API pandas requests tqdm beautifulsoup4

In [ ]:
# import dependencies
import random
import time

import pandas as pd

import requests
import wikipediaapi

import re

from tqdm import tqdm

from urllib.parse import unquote, urljoin, urlparse

from bs4 import BeautifulSoup

## Using Wikidata to get a Clean List of Politicians

[Wikidata](https://www.wikidata.org/wiki/Wikidata:Main_Page) is the structured database behind Wikipedia. Instead of parsing a messy Wikipedia table, we can write a SPARQL query to ask Wikidata directly for: 

*"Give me all people whose position held is Member of the European Parliament for the 10th term, and give me their English Wikipedia article URL and optionally if they have their own website, their website url."*


If you want to try out more queries and examples, you can check the [official query page](https://query.wikidata.org/#).

In [ ]:
# Define our User-Agent (Required by Wikimedia Foundation)
USER_AGENT = "DataCleaningClass (maximilian.kreutner@uni-mannheim.de)"

# SPARQL query to get current MEPs (10th term = Q114425478)
sparql_query = """
SELECT ?person ?personLabel ?article ?officialWebsite WHERE {
  ?person p:P39 ?statement .
  ?statement ps:P39 wd:Q27169 ;              # Member of the European Parliament
             pq:P2937 wd:Q114425478 .        # Tenth European Parliament

  OPTIONAL { ?person wdt:P856 ?officialWebsite . }   # official website
  ?article schema:about ?person ;
           schema:isPartOf <https://en.wikipedia.org/> .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". } # We only want the english articles
}
ORDER BY ?person
"""

title = "https://query.wikidata.org/sparql"
headers = {
    "User-Agent": USER_AGENT,
    "Accept": "application/sparql-results+json"
}

response = requests.get(
    title,
    headers=headers,
    params={"query": sparql_query, "format": "json"},
    timeout=60
)

response.raise_for_status()
data = response.json()
data

We get back a json. Fill that data into a DataFrame which contains the columns: `name`, `official_website`, `wiki_title` and `URL`.
The wiki title is the last part of the URL after the last `/`.

In [ ]:
mep_list = []

# TODO Your code here

df = pd.DataFrame(mep_list)
display(df)

Let's do cleaning steps:

*Check for duplicates and remove them*

*Get insight into how many web adresses we have*

Check if we have an Wikipedia article and an official website for our politicians.

## Fetch the actual full text articles

We can utilize `wikipediaapi` to get the full text of all the articles. For this we use the entries in `wiki_title` of our DataFrame.

To get the text of a wiki page we can use `Wikipedia.page(title)` and then `page.text`.

Save it in the DataFrame in the column `raw_text`.

We will limit the amount of politicians to 20, who have a website for this tutorial.

In [ ]:
df_with_website = df[df["official_website"].notna()].head(20).reset_index()

# Initialize the API
wiki = wikipediaapi.Wikipedia(user_agent=USER_AGENT, language='en')

# TODO Your code here

We still see some section headers that we would like to avoid, e.g. References.

## Text Cleaning

The text usually contains artifacts that we don't want as a description of politicians, e.g. `References`, `External Links`, `See also`, `further reading` and `notes`.

We also should remove trailing whitespaces and normalize whitespaces that contain multiple characters.

Clean the text and safe it in the column `clean_text`.

In [ ]:
import re
import pandas as pd

def clean_wiki_text(text):
    # TODO your code here
    return text

df_with_website["clean_text"] = df_with_website["raw_text"].apply(clean_wiki_text)

## Find social media accounts on the official websites

We can use beautifulsoup and requests to find social media links on the official websites of the MEP.

Implement a method that takes the url and returns a dictionary with the `youtube`, `instagram`, `x_com` and `facebook` account of the politician.

In [ ]:
def find_social_links(website_url):
    result = {
        "youtube": None,
        "instagram": None,
        "x_com": None,
        "facebook": None,
    }

    #TODO your code here

    return result

In [ ]:
df_with_website[["youtube", "instagram", "x_com", "facebook"]] = (
    df_with_website["official_website"]
    .apply(find_social_links)
    .apply(pd.Series)
)

In [ ]:
df_with_website